# Question 3: Spelling Corrector (Non-word and Real-word errors, edit distance 1)

This notebook builds a spelling corrector using the Brown corpus. I am comparing two ways
of generating candidate corrections (plain edit distance 1, and symmetric delete / SymSpell
style), using them for both non-word and real-word error correction, running a small
accuracy evaluation, doing a speed benchmark between the two candidate generation methods,
and finally wrapping everything into a small interactive command line loop.


In [1]:
import nltk
nltk.download('brown', quiet=True)

import re
import time
import random
import string
from collections import Counter, defaultdict

from nltk.corpus import brown

random.seed(42)


## Part 1: Corpus and Model Preparation

I am pulling all sentences from Brown, lowercasing everything, and keeping only tokens made
up purely of letters (so things like punctuation and numbers get dropped, since they are not
really relevant to spelling correction). I split off a held out test portion up front so that
none of the error injection or evaluation later on touches training data.


In [2]:
raw_sents = brown.sents()

sents = []
for s in raw_sents:
    cleaned = [w.lower() for w in s if re.fullmatch(r"[a-zA-Z]+", w)]
    if len(cleaned) > 1:
        sents.append(cleaned)

random.shuffle(sents)

split_point = int(0.9 * len(sents))
train_sents = sents[:split_point]
test_sents = sents[split_point:]

print("total sentences:", len(sents))
print("train sentences:", len(train_sents))
print("test sentences (10%):", len(test_sents))


total sentences: 55820
train sentences: 50238
test sentences (10%): 5582


In [3]:
# unigram frequencies (vocabulary + counts)
unigram_counts = Counter()
for s in train_sents:
    unigram_counts.update(s)

vocab = set(unigram_counts.keys())
total_word_count = sum(unigram_counts.values())

def unigram_prob(word):
    return unigram_counts.get(word, 0) / total_word_count

print("vocab size:", len(vocab))
print("total word tokens:", total_word_count)
print("example frequency, 'the':", unigram_counts['the'])


vocab size: 38652
total word tokens: 883748
example frequency, 'the': 63169


In [4]:
# bigram counts, used for real-word error correction (context matters here)
bigram_counts = defaultdict(Counter)
for s in train_sents:
    for w1, w2 in zip(s, s[1:]):
        bigram_counts[w1][w2] += 1

VOCAB_SIZE = len(vocab)

def bigram_prob(w1, w2, k=1):
    # add-k smoothing so unseen bigrams do not just get probability 0
    denom = unigram_counts.get(w1, 0) + k * VOCAB_SIZE
    return (bigram_counts[w1][w2] + k) / denom

print(bigram_prob('of', 'the'))
print(bigram_prob('purple', 'elephant'))


0.12322870957824422
2.5865190626454916e-05


## Part 2: Candidate Generation

### Method A: standard edit distance 1

This is the classic approach, generate every possible deletion, transposition, replacement
and insertion of the word, then keep whichever of those strings happen to be real vocabulary
words.


In [5]:
alphabet = string.ascii_lowercase

def edits1(word):
    splits = [(word[:i], word[i:]) for i in range(len(word) + 1)]
    deletes = [L + R[1:] for L, R in splits if R]
    transposes = [L + R[1] + R[0] + R[2:] for L, R in splits if len(R) > 1]
    replaces = [L + c + R[1:] for L, R in splits if R for c in alphabet]
    inserts = [L + c + R for L, R in splits for c in alphabet]
    return set(deletes + transposes + replaces + inserts)

def method_a_candidates(word):
    return {w for w in edits1(word) if w in vocab}

print(method_a_candidates('helo'))


{'halo', 'held', 'hero', 'helm', 'hell', 'hilo', 'help', 'hel', 'hello'}


### Method B: symmetric delete (SymSpell style)

The idea here is to shift almost all of the work to preprocessing. For every vocabulary word
I generate all of its one character deletions and store them in a dictionary that maps each
deletion back to the original word(s) it came from. Then at query time, I only need to
generate the deletions of the misspelled word itself (much cheaper than generating deletes,
inserts, replaces and transposes against the whole alphabet) and look those up.


In [6]:
def one_char_deletes(word):
    if len(word) <= 1:
        return {""}
    return {word[:i] + word[i + 1:] for i in range(len(word))}

symspell_dict = defaultdict(set)
for w in vocab:
    symspell_dict[w].add(w)
    for d in one_char_deletes(w):
        symspell_dict[d].add(w)

print("symspell dictionary entries:", len(symspell_dict))


symspell dictionary entries: 298049


In [7]:
def method_b_candidates(word):
    candidates = set(symspell_dict.get(word, set()))
    for d in one_char_deletes(word):
        candidates |= symspell_dict.get(d, set())
    candidates.discard('')
    return candidates

print(method_b_candidates('helo'))


{'halo', 'heel', 'held', 'hero', 'eloi', 'helm', 'heal', 'hell', 'hilo', 'help', 'hel', 'hello'}


In [8]:
# quick sanity check, both methods should broadly agree on simple typos
for test_word in ['helo', 'wrold', 'speling', 'recieve']:
    a = method_a_candidates(test_word)
    b = method_b_candidates(test_word)
    print(test_word, '-> A:', a, '| B:', b)


helo -> A: {'halo', 'held', 'hero', 'helm', 'hell', 'hilo', 'help', 'hel', 'hello'} | B: {'halo', 'heel', 'held', 'hero', 'eloi', 'helm', 'heal', 'hell', 'hilo', 'help', 'hel', 'hello'}
wrold -> A: {'wold', 'world'} | B: {'would', 'wolde', 'wold', 'world'}
speling -> A: {'spelling'} | B: {'sealing', 'spelling', 'sapling', 'pelting', 'selling', 'peeling'}
recieve -> A: {'relieve', 'receive'} | B: {'relieve', 'receive', 'receave'}


## Part 3: Spelling Correction Logic

### Non-word errors

If a word is not in the vocabulary at all, I pull candidates from both methods, merge them,
and pick whichever candidate has the highest unigram frequency (the most common explanation
for a typo is usually the more common word).


In [9]:
def correct_nonword(word, method='both'):
    candidates = set()
    if method in ('a', 'both'):
        candidates |= method_a_candidates(word)
    if method in ('b', 'both'):
        candidates |= method_b_candidates(word)

    if not candidates:
        return word  # nothing to suggest, just return as is

    return max(candidates, key=lambda w: unigram_counts[w])

print(correct_nonword('helo'))
print(correct_nonword('wrold'))


help
would


### Real-word errors

Here the typed word is already a valid word, so frequency alone is not enough (both "sea"
and "see" are common). Instead I look at the surrounding context using the bigram model,
compare the probability of the phrase as typed against the probability of the phrase with
a nearby candidate substituted in, and only flip to the candidate if it scores meaningfully
higher (using a threshold so I do not swap words for a marginal difference).


In [10]:
def correct_realword(prev_word, word, next_word=None, method='both', threshold=2.0):
    if word not in vocab:
        return word  # not a real-word case

    candidates = set()
    if method in ('a', 'both'):
        candidates |= method_a_candidates(word)
    if method in ('b', 'both'):
        candidates |= method_b_candidates(word)
    candidates.discard(word)

    if not candidates:
        return word

    def phrase_score(w):
        score = 1e-12  # avoid a hard zero
        if prev_word is not None:
            score *= bigram_prob(prev_word, w)
        if next_word is not None:
            score *= bigram_prob(w, next_word)
        return score

    original_score = phrase_score(word)
    best_word = word
    best_score = original_score

    for c in candidates:
        c_score = phrase_score(c)
        if c_score > best_score * threshold:
            best_score = c_score
            best_word = c

    return best_word

print(correct_realword('the', 'sea', 'world'))
print(correct_realword('i', 'ate', 'an'), '(context: i ate an apply)')


use
at (context: i ate an apply)


## Part 4: Evaluation and Speed Demon Benchmark

### Building the test set

I use the held out 10% of Brown sentences from Part 1. For each sentence, I pick one word at
random and corrupt it with a single random edit. I build two versions of each corrupted
sentence, one where the corruption happens to land outside the vocabulary (a non-word error),
and one where it happens to land on a different valid vocabulary word (a real-word error).


In [11]:
def apply_random_edit(word):
    if len(word) < 2:
        return word
    op = random.choice(['delete', 'insert', 'substitute', 'transpose'])
    i = random.randrange(len(word))

    if op == 'delete':
        return word[:i] + word[i + 1:]
    elif op == 'insert':
        c = random.choice(alphabet)
        return word[:i] + c + word[i:]
    elif op == 'substitute':
        c = random.choice(alphabet)
        return word[:i] + c + word[i + 1:]
    else:  # transpose
        j = min(i + 1, len(word) - 1)
        chars = list(word)
        chars[i], chars[j] = chars[j], chars[i]
        return ''.join(chars)


In [12]:
def make_nonword_example(sent, max_tries=20):
    idx = random.randrange(len(sent))
    original = sent[idx]
    corrupted = original
    for _ in range(max_tries):
        candidate = apply_random_edit(original)
        if candidate != original and candidate not in vocab:
            corrupted = candidate
            break
    new_sent = sent.copy()
    new_sent[idx] = corrupted
    return new_sent, idx, original


def make_realword_example(sent, max_tries=20):
    idx = random.randrange(len(sent))
    original = sent[idx]
    for _ in range(max_tries):
        candidate = apply_random_edit(original)
        if candidate != original and candidate in vocab:
            new_sent = sent.copy()
            new_sent[idx] = candidate
            return new_sent, idx, original
    return None  # could not find a real-word substitute for this word, skip it


In [13]:
nonword_test_set = []
realword_test_set = []

for s in test_sents:
    if len(s) < 2:
        continue
    nonword_test_set.append(make_nonword_example(s))
    rw = make_realword_example(s)
    if rw is not None:
        realword_test_set.append(rw)

print("non-word test examples:", len(nonword_test_set))
print("real-word test examples:", len(realword_test_set))


non-word test examples: 5582
real-word test examples: 3447


### Accuracy

In [14]:
def evaluate_nonword(test_set, method='both'):
    correct = 0
    for corrupted_sent, idx, original in test_set:
        prediction = correct_nonword(corrupted_sent[idx], method=method)
        if prediction == original:
            correct += 1
    return correct / len(test_set)


def evaluate_realword(test_set, method='both'):
    correct = 0
    for corrupted_sent, idx, original in test_set:
        prev_w = corrupted_sent[idx - 1] if idx > 0 else None
        next_w = corrupted_sent[idx + 1] if idx < len(corrupted_sent) - 1 else None
        prediction = correct_realword(prev_w, corrupted_sent[idx], next_w, method=method)
        if prediction == original:
            correct += 1
    return correct / len(test_set)


nonword_acc = evaluate_nonword(nonword_test_set)
realword_acc = evaluate_realword(realword_test_set)

print(f"Non-word error correction accuracy: {nonword_acc:.2%}")
print(f"Real-word error correction accuracy: {realword_acc:.2%}")


Non-word error correction accuracy: 72.66%
Real-word error correction accuracy: 59.01%


### Speed Demon Benchmark

Now I isolate just the candidate generation step (the non-word correction logic) and compare
Method A against Method B on a fixed batch of exactly 1000 misspelled words.


In [15]:
misspelled_batch = []
attempts = 0
vocab_list = list(vocab)

while len(misspelled_batch) < 1000 and attempts < 50000:
    attempts += 1
    w = random.choice(vocab_list)
    corrupted = apply_random_edit(w)
    if corrupted != w and corrupted not in vocab:
        misspelled_batch.append(corrupted)

print("batch size:", len(misspelled_batch))


batch size: 1000


In [16]:
start = time.perf_counter()
for w in misspelled_batch:
    method_a_candidates(w)
time_a = time.perf_counter() - start

start = time.perf_counter()
for w in misspelled_batch:
    method_b_candidates(w)
time_b = time.perf_counter() - start

print(f"Method A (standard edit distance 1) total time: {time_a:.4f} s")
print(f"Method B (symmetric delete)        total time: {time_b:.4f} s")
print(f"Method B is roughly {time_a / time_b:.1f}x faster on this batch")


Method A (standard edit distance 1) total time: 0.0717 s
Method B (symmetric delete)        total time: 0.0053 s
Method B is roughly 13.6x faster on this batch


**Conclusion on the speed difference:**

Method A has to build every possible deletion, transposition, replacement and insertion of
the misspelled word from scratch on every single call, and the replace/insert steps loop
over all 26 letters at every position, so the amount of work per query scales with word
length times alphabet size, and then every one of those generated strings still has to be
checked against the vocabulary set. Method B pushes almost all of that cost into a one-time
preprocessing pass over the vocabulary (building the deletes dictionary), so at query time it
only has to generate the deletions of the misspelled word itself, which is a much smaller set
that scales only with word length, and then does simple dictionary lookups. That is exactly
why Method B ends up noticeably faster once the batch size is large, the expensive part of
the work already happened before any of the 1000 words were even seen.


## Part 5: Live Interactive Application

This wraps the correction logic into a small command line loop. It corrects each word in a
typed sentence (running non-word correction for out of vocabulary words and real-word
correction for in-vocabulary words using their neighbors), wraps any word that got changed in
asterisks, and prints how long the correction took. Typing `exit` ends the loop.

This is written so it can be run either from a plain Python script in a terminal, or directly
inside a notebook cell, since `input()` works fine in both.


In [17]:
def correct_sentence(sentence, method='both'):
    raw_tokens = sentence.split()
    clean_tokens = [re.sub(r'[^a-zA-Z]', '', t).lower() for t in raw_tokens]

    corrected = []
    changed = []

    for i, w in enumerate(clean_tokens):
        if not w:
            corrected.append(raw_tokens[i])
            changed.append(False)
            continue

        if w not in vocab:
            fixed = correct_nonword(w, method=method)
        else:
            prev_w = clean_tokens[i - 1] if i > 0 else None
            next_w = clean_tokens[i + 1] if i + 1 < len(clean_tokens) else None
            fixed = correct_realword(prev_w, w, next_w, method=method)

        corrected.append(fixed)
        changed.append(fixed != w)

    return corrected, changed


In [18]:
def run_spelling_corrector_cli():
    print("Spelling corrector, type a sentence and press enter.")
    print("Changed words are shown in **asterisks**. Type 'exit' to quit.")
    while True:
        sentence = input("> ")
        if sentence.strip().lower() == 'exit':
            print("Goodbye")
            break
        if not sentence.strip():
            continue

        start = time.perf_counter()
        corrected_words, changed_flags = correct_sentence(sentence)
        elapsed_ms = (time.perf_counter() - start) * 1000

        display_words = []
        for w, was_changed in zip(corrected_words, changed_flags):
            display_words.append(f"**{w}**" if was_changed else w)

        print(' '.join(display_words))
        print(f"(corrected in {elapsed_ms:.2f} ms)")

# uncomment the line below to actually run the interactive loop in this notebook
# run_spelling_corrector_cli()


### Trying it on the example sentences from the assignment

I am calling `correct_sentence` directly here instead of the full CLI loop, just so the
notebook output is visible without needing manual input.


In [19]:
example_sentences = [
    "I hav a good feeling about this.",
    "This is a test sentnce.",
    "I would like to sea the world.",
    "Please meat me at the station.",
]

for sent in example_sentences:
    corrected_words, changed_flags = correct_sentence(sent)
    display_words = [f"**{w}**" if c else w for w, c in zip(corrected_words, changed_flags)]
    print("input :", sent)
    print("output:", ' '.join(display_words))
    print()


input : I hav a good feeling about this.
output: i **had** a good feeling about this

input : This is a test sentnce.
output: this is a test **sentence**

input : I would like to sea the world.
output: **it** would like to **see** the world

input : Please meat me at the station.
output: please **beat** **be** **to** the station

